# Task 1 — Solution Analysis
Visualisation and analysis of the Gurobi optimisation results.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS = '../data/results/task_1'

summary    = json.load(open(f'{RESULTS}/summary.json'))
expansions = pd.read_csv(f'{RESULTS}/expansions.csv')
zip_res    = pd.read_csv(f'{RESULTS}/zip_results.csv')

# Only keep facilities that were actually expanded
expanded = expansions[expansions['x_total'] > 0].copy()

print(f"Facilities expanded       : {len(expanded):,}")
print(f"Desert zip codes          : {len(zip_res):,}")
print(f"Total cost                : ${summary['total_cost']:,.0f}")

## 1. Cost Breakdown

In [ ]:
labels = ['Expansion\nCost', 'Construction\nCost', 'Equipment\nSurcharge']
values = [summary['C_exp'], summary['C_build'], summary['C_equip']]
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie
axes[0].pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
axes[0].set_title(f'Total Cost: ${sum(values):,.0f}', fontsize=13, fontweight='bold')

# Bar
bars = axes[1].bar(labels, [v/1e6 for v in values], color=colors, edgecolor='white')
axes[1].set_ylabel('Cost ($ millions)')
axes[1].set_title('Cost Components', fontsize=13)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'${val/1e6:.1f}M', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 2. Expansion Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of expansion sizes
axes[0].hist(expanded['x_total'], bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Slots Added per Facility')
axes[0].set_ylabel('Number of Facilities')
axes[0].set_title('Distribution of Expansion Sizes')

# Cheap vs expensive tier
tier_counts = expanded['tier'].value_counts()
axes[1].bar(tier_counts.index, tier_counts.values,
            color=['#55A868', '#C44E52'], edgecolor='white')
axes[1].set_ylabel('Number of Facilities')
axes[1].set_title('Expansion Cost Tier\n(cheap: x < n_f  |  expensive: x ≥ n_f)')
for i, (idx, val) in enumerate(tier_counts.items()):
    axes[1].text(i, val + 0.5, str(val), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# 0-5 vs 5-12 expansion split
fig, ax = plt.subplots(figsize=(7, 5))

age_totals = {
    '0–5 slots':  summary['expansion_slots_05'],
    '5–12 slots': summary['expansion_slots_512'],
}
bars = ax.bar(age_totals.keys(), age_totals.values(), color=['#4C72B0', '#DD8452'], edgecolor='white')
ax.set_ylabel('Total Slots Added')
ax.set_title('Expansion Slots by Age Group')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, val in zip(bars, age_totals.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,.0f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 3. New Facilities Built

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Size breakdown
size_labels = ['Small (100)', 'Medium (200)', 'Large (400)']
size_vals   = [summary['new_facs_S'], summary['new_facs_M'], summary['new_facs_L']]
colors = ['#8172B3', '#4C72B0', '#55A868']
bars = axes[0].bar(size_labels, size_vals, color=colors, edgecolor='white')
axes[0].set_ylabel('Facilities Built')
axes[0].set_title('New Facilities by Size')
for bar, val in zip(bars, size_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(int(val)), ha='center', fontsize=11)

# Zips that got new builds vs expansion only
got_build = (zip_res['total_new_facs'] > 0).sum()
got_exp   = (zip_res[['exp_slots_05','exp_slots_512']].sum(axis=1) > 0).sum()
only_exp  = ((zip_res['total_new_facs'] == 0) & (zip_res[['exp_slots_05','exp_slots_512']].sum(axis=1) > 0)).sum()
both      = ((zip_res['total_new_facs'] > 0) & (zip_res[['exp_slots_05','exp_slots_512']].sum(axis=1) > 0)).sum()
only_build = got_build - both

cat_labels = ['Expansion only', 'New build only', 'Both']
cat_vals   = [only_exp, only_build, both]
axes[1].bar(cat_labels, cat_vals, color=['#4C72B0','#DD8452','#C44E52'], edgecolor='white')
axes[1].set_ylabel('Zip Codes')
axes[1].set_title('How Desert Zips Were Addressed')
for i, val in enumerate(cat_vals):
    axes[1].text(i, val + 0.1, str(val), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 4. Per-Zip Deficit vs Slots Added

In [ ]:
zip_res['total_slots_added'] = (
    zip_res['exp_slots_05'] + zip_res['exp_slots_512'] +
    zip_res['n_05'] + zip_res['n_512']
)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    zip_res['total_deficit'], zip_res['total_slots_added'],
    c=(zip_res['demand'] == 'High').astype(int),
    cmap='coolwarm', alpha=0.7, edgecolors='white', linewidths=0.5, s=60
)
# 1:1 line
mx = max(zip_res['total_deficit'].max(), zip_res['total_slots_added'].max())
ax.plot([0, mx], [0, mx], 'k--', linewidth=0.8, label='1:1 line')
ax.set_xlabel('Total Slot Deficit')
ax.set_ylabel('Total Slots Added')
ax.set_title('Deficit vs Slots Added per Desert Zip')
ax.legend(['1:1 line', 'Normal demand', 'High demand'], loc='upper left')
plt.colorbar(sc, ax=ax, label='High demand (1) / Normal (0)')
plt.tight_layout()
plt.show()

## 5. Top 10 Zip Codes by Slots Added

In [ ]:
top10 = zip_res.nlargest(10, 'total_slots_added')[[
    'zipcode','demand','total_deficit','under5_deficit',
    'exp_slots_05','exp_slots_512','n_05','n_512','total_slots_added'
]].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(top10))
w = 0.35
ax.bar([i - w/2 for i in x],
       top10['exp_slots_05'] + top10['exp_slots_512'],
       w, label='Expansion slots', color='#4C72B0')
ax.bar([i + w/2 for i in x],
       top10['n_05'] + top10['n_512'],
       w, label='New build slots', color='#DD8452')
ax.set_xticks(list(x))
ax.set_xticklabels(top10['zipcode'].astype(str), rotation=45)
ax.set_ylabel('Slots Added')
ax.set_title('Top 10 Zip Codes by Total Slots Added')
ax.legend()
plt.tight_layout()
plt.show()

top10